In [2]:
print("Length of X_train['processed_text']:", len(X_train['processed_text']))
print("Length of y_train_encoded:", len(y_train_encoded))


Length of X_train['processed_text']: 87095
Length of y_train_encoded: 87095


In [1]:
import joblib
from datasets import Dataset

# Load the processed data and encoded labels
X_train = joblib.load('data/X_train.pkl')
X_val = joblib.load('data/X_val.pkl')
y_train_encoded = joblib.load('data/y_train_encoded.pkl')
y_val_encoded = joblib.load('data/y_val_encoded.pkl')

# Confirm that the labels are integers
print("Encoded labels:", y_train_encoded[:5])  # Should output integers

# Create train_data and val_data with encoded labels
train_data = Dataset.from_dict({"text": X_train['processed_text'], "labels": y_train_encoded})
val_data = Dataset.from_dict({"text": X_val['processed_text'], "labels": y_val_encoded})


Encoded labels: [ 4 15 21 22 15]


In [4]:
# Display the first few entries in train_data and val_data to confirm labels are present
print(train_data[0])
print(val_data[0])


{'text': 'got scam app name koko loan transfer money account without even concern transfer rupe show pay back next day sixth day today still time pay amount start receiv threaten call messag sinc yesterday blackmail use contact list blackmail personsi realli worri scare', 'labels': 4}
{'text': 'infin cash compani negit compani compani', 'labels': 14}


In [3]:
import joblib
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load processed data and encoded labels
X_train = joblib.load('data/X_train.pkl')
X_val = joblib.load('data/X_val.pkl')
y_train_encoded = joblib.load('data/y_train_encoded.pkl')
y_val_encoded = joblib.load('data/y_val_encoded.pkl')
X_train = joblib.load('data/X_train.pkl')
X_val = joblib.load('data/X_val.pkl')
y_train = joblib.load('data/y_train.pkl')
y_val = joblib.load('data/y_val.pkl')
print("Data loaded successfully.")



Data loaded successfully.


In [8]:
print(train_data[0]['labels'])  # Should output an integer
print(val_data[0]['labels'])    # Should output an integer


4
14


In [5]:
# Model Training with BERT - Cyberfraud Classifier
# This notebook handles the training of a BERT model for classifying cyberfraud types with balanced class weights.

# Import necessary libraries
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
import joblib
import numpy as np

# Step 1: Set Device
# Acknowledgment: Determine if a GPU is available for faster training; default to CPU otherwise.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Step 2: Load the Processed Data
# Acknowledgment: Load the balanced training and validation data created during the preprocessing phase.
X_train = joblib.load('data/X_train.pkl')
X_val = joblib.load('data/X_val.pkl')
y_train = joblib.load('data/y_train_encoded.pkl')
y_val = joblib.load('data/y_val_encoded.pkl')

# Step 3: Calculate Class Weights
# Acknowledgment: Calculate class weights based on the balanced training labels to address class imbalance during model training.
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights calculated for training:", class_weights_tensor)

# Step 4: Prepare Dataset for BERT
# Acknowledgment: Create train and validation datasets for BERT, including processed text and encoded labels.
train_data = Dataset.from_dict({"text": X_train['processed_text'], "labels": y_train})
val_data = Dataset.from_dict({"text": X_val['processed_text'], "labels": y_val})

# Step 5: Tokenize Text Data
# Acknowledgment: Tokenize text data using BERT tokenizer and set format for PyTorch compatibility.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize_function, batched=True)
val_data = val_data.map(tokenize_function, batched=True)

# Set the data format for PyTorch
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Step 6: Load BERT Model for Sequence Classification
# Acknowledgment: Initialize BERT model for sequence classification with a number of labels equal to unique classes in y_train.
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(np.unique(y_train))).to(device)

# Step 7: Define Training Arguments
# Acknowledgment: Set up training arguments to optimize for faster training with minimal disk usage.
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy="epoch",                # Evaluate at the end of each epoch
    save_strategy="no",                         # Disable checkpoint saving to conserve space
    num_train_epochs=2,                         # Reduced to 2 epochs for faster training
    per_device_train_batch_size=16,             # Larger batch size for faster training
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,              # Gradient accumulation for effective larger batch size
    fp16=True,                                  # Mixed precision for faster training
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
)

# Step 8: Custom Trainer Class with Weighted Loss
# Acknowledgment: Define a custom Trainer class to apply class weights in the loss function, improving performance on underrepresented classes.
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):  # Added **kwargs here to handle unexpected arguments
        labels = inputs.get("labels").to(device)
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Apply class weights in the loss calculation
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# Step 9: Initialize Trainer with Training and Validation Data
# Acknowledgment: Set up the Trainer instance with training arguments, datasets, and metrics for model evaluation.
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics
)

# Step 10: Train the Model
# Acknowledgment: Start the training process with optimized settings, skipping intermediate checkpoints to save space.
trainer.train()

print("Training complete.")


Using device: cuda
Class weights calculated for training: tensor([1.0012, 1.0015, 1.0011, 0.9946, 1.0074, 1.0004, 1.0005, 0.9985, 0.9957,
        0.9999, 0.9958, 0.9996, 0.9988, 1.0004, 0.9976, 0.9984, 0.9949, 1.0027,
        0.9930, 1.0047, 1.0071, 1.0012, 1.0024, 0.9992, 0.9995, 1.0002, 1.0002,
        0.9991, 0.9994, 1.0044, 1.0042, 0.9948, 1.0012, 0.9983, 1.0025],
       device='cuda:0')


Map:   0%|          | 0/751968 [00:00<?, ? examples/s]

Map:   0%|          | 0/187992 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\transformers\training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.276100,0.569955,0.795752,0.793056,0.797985,0.795752
2,0.596800,0.212527,0.937896,0.937401,0.938627,0.937896


Training complete.


In [2]:
# Step 1: Save the trained model weights
# Acknowledgment: Saving model weights to avoid retraining and allow for future use in evaluation.
# This will save the model in './results/bert_cyberfraud_classifier'
model.save_pretrained('./results/bert_cyberfraud_classifier')
print("Model weights saved successfully in './results/bert_cyberfraud_classifier'.")

# Step 2: Verify that the model and tokenizer files are saved correctly.
# Acknowledgment: Check that 'pytorch_model.bin' and tokenizer files exist in the specified path.
# You can list the contents or manually check to confirm.

# Step 3: Load the model with saved weights for evaluation
# Acknowledgment: Loading the model from saved weights to proceed directly to evaluation.
from transformers import BertForSequenceClassification

# Load model from saved path
model = BertForSequenceClassification.from_pretrained('./results/bert_cyberfraud_classifier')
print("Model loaded from './results/bert_cyberfraud_classifier' for evaluation.")


NameError: name 'model' is not defined

In [4]:
# Save the model
model.save_pretrained('./results/bert_cyberfraud_classifier')
tokenizer.save_pretrained('./results/bert_cyberfraud_classifier')


NameError: name 'model' is not defined

In [2]:
import joblib

# Load preprocessed training and test data
X_train = joblib.load('data/X_train.pkl')
y_train = joblib.load('data/y_train_encoded.pkl')
X_test = joblib.load('data/X_test.pkl')
y_test = joblib.load('data/y_test_encoded.pkl')

print("Data loaded successfully.")


Data loaded successfully.


In [6]:
import os

# Check if the model directory contains expected files
model_dir = './results/bert_cyberfraud_classifier'
model_files = os.listdir(model_dir)
print("Files in saved model directory:", model_files)


Files in saved model directory: ['config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.txt']


In [5]:
# Number of labels expected in training
num_labels_in_training = 34  # Adjust based on your training set configuration

# Check if all labels in y_test_encoded are within the expected range
min_label = y_test_encoded.min()
max_label = y_test_encoded.max()

if min_label >= 0 and max_label < num_labels_in_training:
    print("All labels are within the expected range.")
else:
    print(f"Warning: Labels out of range. Min label: {min_label}, Max label: {max_label}")


In [4]:
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd


# Step 1: Initialize and fit label encoder on training set labels
# Acknowledgment: Consistent encoding on both train and test sets
label_encoder = LabelEncoder()
label_encoder.fit(train_df['sub_category'])

# Encode training labels and save the encoder
y_train_encoded = label_encoder.transform(train_df['sub_category'])
joblib.dump(label_encoder, 'label_encoder.pkl')  # Save encoder for future use

# Step 2: Replace unknown labels in the test set
# Fill NaNs with the most frequent category in the training data
test_df['sub_category'] = test_df['sub_category'].fillna(train_df['sub_category'].mode()[0])

# Ensure only labels present in the encoder are in test_df['sub_category']
valid_categories = set(label_encoder.classes_)
test_df = test_df[test_df['sub_category'].isin(valid_categories)]

# Encode test labels with the same encoder
y_test_encoded = label_encoder.transform(test_df['sub_category'])

# Step 3: Verify label range
# Acknowledgment: Check that re-encoded labels fall within expected range
min_label = y_test_encoded.min()
max_label = y_test_encoded.max()
num_labels_in_training = len(label_encoder.classes_)

if min_label >= 0 and max_label < num_labels_in_training:
    print("All labels are within the expected range.")
else:
    print(f"Warning: Labels out of range. Min label: {min_label}, Max label: {max_label}")

# Step 4: Save processed test data for evaluation
# Acknowledgment: Save transformed test data for model evaluation
X_test = test_df[['processed_text']]
joblib.dump(X_test, 'data/test_data.pkl')
joblib.dump(y_test_encoded, 'data/y_test_encoded.pkl')
print("Processed test data and labels saved successfully.")


All labels are within the expected range.
Processed test data and labels saved successfully.


In [3]:
import pandas as pd
import re
import joblib
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import BertTokenizer

# Load the training data to access label encoding
train_df = pd.read_csv('data/train.csv')  # Adjust the path if needed

# Load the test data CSV
test_df = pd.read_csv('data/test.csv')  # Adjust the path if needed

# Preprocess text in the test data
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

test_df['processed_text'] = test_df['crimeaditionalinfo'].apply(lambda x: preprocess_text(str(x)))

# Fill missing values in 'sub_category' with the most frequent category
most_frequent_category = train_df['sub_category'].mode()[0]
test_df['sub_category'] = test_df['sub_category'].fillna(most_frequent_category)

# Fit the label encoder on training data labels only
label_encoder = LabelEncoder()
label_encoder.fit(train_df['sub_category'])

# Remove rows from test_df with unseen labels
seen_labels = set(label_encoder.classes_)
test_df = test_df[test_df['sub_category'].isin(seen_labels)]

# Encode the labels
y_test_encoded = label_encoder.transform(test_df['sub_category'])

# Save the processed test data and encoded labels
joblib.dump(test_df[['processed_text']], 'data/test_data.pkl')
joblib.dump(y_test_encoded, 'data/y_test_encoded.pkl')
print("Test data and labels saved without unseen labels.")


Test data and labels saved without unseen labels.


In [1]:
import pandas as pd

# Reload the training dataset CSV
train_df = pd.read_csv('data/train.csv')  # Adjust the path if necessary

# Preprocess training data if needed
# Assuming 'sub_category' and 'processed_text' columns were part of the training data
train_df['sub_category'] = train_df['sub_category'].fillna(train_df['sub_category'].mode()[0])

# If 'processed_text' column is needed and was preprocessed earlier, preprocess or load the processed version
# train_df['processed_text'] = ...

print("Training data loaded successfully.")
test_df = pd.read_csv('data/test.csv')  # Adjust the path if necessary

# Preprocess test data if needed
# Example of handling missing 'sub_category' labels and processing 'crimeaditionalinfo' column
test_df['sub_category'] = test_df['sub_category'].fillna(train_df['sub_category'].mode()[0])

# If 'processed_text' was processed earlier, apply the processing function again or load the preprocessed data
# Assuming you have a 'processed_text' column, preprocess or transform it as needed
# test_df['processed_text'] = test_df['crimeaditionalinfo'].apply(preprocess_text)

print("Test data loaded and preprocessed successfully.")

Training data loaded successfully.
Test data loaded and preprocessed successfully.


In [6]:
import joblib
import pandas as pd
import numpy as np
import torch
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

# Set device to CUDA if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Step 1: Reload test data and perform preprocessing
# Acknowledgment: Load test data from CSV and apply preprocessing for consistency with training data
test_df = pd.read_csv('data/test.csv')  # Adjust path if necessary

# Fill missing labels in test_df with the most frequent category from test_df
most_frequent_category = test_df['sub_category'].mode()[0]
test_df['sub_category'] = test_df['sub_category'].fillna(most_frequent_category)

# Preprocess the text data
def preprocess_text(text):
    import re
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

test_df['processed_text'] = test_df['crimeaditionalinfo'].apply(lambda x: preprocess_text(str(x)))

# Load the training data to fit LabelEncoder on its labels
train_df = pd.read_csv('data/train.csv')  # Adjust path if necessary
label_encoder = LabelEncoder()
label_encoder.fit(train_df['sub_category'])  # Fit encoder on training labels

# Add an 'Unknown' class at the end of the label encoder for any unseen labels
label_encoder.classes_ = np.append(label_encoder.classes_, 'Unknown')
test_df['sub_category'] = test_df['sub_category'].apply(lambda x: x if x in label_encoder.classes_ else 'Unknown')

# Transform the labels
y_test_encoded = label_encoder.transform(test_df['sub_category'])

# Ensure labels start from 0 and are sequential
num_labels_in_training = len(label_encoder.classes_)
y_test_encoded = np.clip(y_test_encoded, 0, num_labels_in_training - 1)
print("Unique labels in y_test_encoded after adjustment:", set(y_test_encoded))

# Save preprocessed test data and encoded labels
joblib.dump(test_df[['processed_text']], 'data/test_data.pkl')
joblib.dump(y_test_encoded, 'data/y_test_encoded.pkl')
print("Test data and labels saved successfully.")

# Step 2: Reload tokenizer and model with saved weights for evaluation
# Acknowledgment: Load tokenizer and trained model weights
tokenizer = BertTokenizer.from_pretrained('./results/bert_cyberfraud_classifier')
model = BertForSequenceClassification.from_pretrained('./results/bert_cyberfraud_classifier',ignore_mismatched_sizes=True).to(device)
# Step 3: Tokenize the test data
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

# Reload preprocessed test data
X_test = joblib.load('data/test_data.pkl')
y_test_encoded = joblib.load('data/y_test_encoded.pkl')

# Convert test data to Dataset and tokenize
test_data = Dataset.from_dict({"text": X_test['processed_text'], "labels": y_test_encoded})
test_data = test_data.map(tokenize_function, batched=True)
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
print("Test data tokenized and formatted.")

# Step 4: Define evaluation arguments and initialize Trainer
training_args = TrainingArguments(
    output_dir='./results',           # Directory for logs
    per_device_eval_batch_size=8,     # Evaluation batch size
    logging_dir='./logs',             # Directory for evaluation logs
    no_cuda=not torch.cuda.is_available()  # Use CPU if CUDA is unavailable
)

trainer = Trainer(
    model=model,
    args=training_args
)

# Step 5: Evaluate model on the test data
# Acknowledgment: Evaluate model performance on test data and print metrics
test_metrics = trainer.evaluate(test_data)
print("Test Set Metrics:", test_metrics)


Using device: cuda
Unique labels in y_test_encoded after adjustment: {np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(36)}
Test data and labels saved successfully.


Map:   0%|          | 0/31229 [00:00<?, ? examples/s]

Test data tokenized and formatted.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [3]:
import os
import joblib
import torch
from transformers import Trainer, TrainingArguments, BertForSequenceClassification, BertTokenizer
from datasets import Dataset

# Set CUDA launch blocking for debugging
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Enforce CPU usage to isolate CUDA issues
device = torch.device("cpu")
print(f"Using device: {device}")

# Load preprocessed test data and labels
X_test = joblib.load('data/test_data.pkl')
y_test_encoded = joblib.load('data/y_test_encoded.pkl')

# Initialize tokenizer and model on CPU
tokenizer = BertTokenizer.from_pretrained('./results/bert_cyberfraud_classifier')
model = BertForSequenceClassification.from_pretrained('./results/bert_cyberfraud_classifier', num_labels=len(set(y_test_encoded))).to(device)

# Prepare test data
test_data = Dataset.from_dict({"text": X_test['processed_text'], "labels": y_test_encoded})
test_data = test_data.map(lambda examples: tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128), batched=True)
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Set up evaluation arguments
training_args = TrainingArguments(
    output_dir='./results',
    per_device_eval_batch_size=8,
    no_cuda=True  # Forces CPU-only
)

# Initialize Trainer for evaluation
trainer = Trainer(
    model=model,
    args=training_args
)

# Run evaluation
test_metrics = trainer.evaluate(test_data)
print("Test Set Metrics:", test_metrics)



Using device: cpu


Map:   0%|          | 0/31225 [00:00<?, ? examples/s]

C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\transformers\training_args.py:1574: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


IndexError: Target 34 is out of bounds.

In [4]:
import torch
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, BertTokenizer
from datasets import Dataset
import os
import joblib
import numpy as np

# Debug settings
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load the saved label encoder and data
label_encoder = joblib.load('data/label_encoder.pkl')
num_labels = len(label_encoder.classes_)

# Load training and test data
X_train = joblib.load('data/X_train.pkl')
y_train = joblib.load('data/y_train_encoded.pkl')
X_test = joblib.load('data/X_test.pkl')
y_test_encoded = joblib.load('data/y_test_encoded.pkl')

# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

# Prepare data function
def prepare_dataset(texts, labels, tokenizer, max_length=512):
    return Dataset.from_dict({
        'text': texts['processed_text'].tolist(),
        'labels': labels.tolist()
    }).map(
        lambda x: tokenizer(
            x['text'],
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ),
        batched=True,
        remove_columns=['text']
    )

# Prepare both training and test datasets
train_dataset = prepare_dataset(X_train, y_train, tokenizer)
test_dataset = prepare_dataset(X_test, y_test_encoded, tokenizer)

# Modified training arguments
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir='./logs',
    no_cuda=False,
    load_best_model_at_end=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none"  # Disable wandb logging if you're not using it
)

# Validation function
def validate_data_labels(labels, num_labels):
    unique_labels = np.unique(labels)
    if max(unique_labels) >= num_labels:
        raise ValueError(f"Found label {max(unique_labels)} which is >= num_labels ({num_labels})")
    if min(unique_labels) < 0:
        raise ValueError(f"Found negative label {min(unique_labels)}")
    print("Label validation passed!")

# Validate labels
validate_data_labels(y_test_encoded, num_labels)

# Training and evaluation with error handling
try:
    print("Starting training and evaluation...")
    print(f"Number of labels in model: {model.config.num_labels}")
    print(f"Number of training examples: {len(train_dataset)}")
    print(f"Number of test examples: {len(test_dataset)}")
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
    )
    
    # Train the model
    print("Training the model...")
    trainer.train()
    
    # Evaluate the model
    print("Evaluating the model...")
    test_results = trainer.evaluate()
    print("Test Results:", test_results)

except Exception as e:
    print(f"Error during training/evaluation: {str(e)}")
    if "CUDA" in str(e):
        print("CUDA error detected, trying CPU...")
        training_args.no_cuda = True
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
        )
        trainer.train()
        test_results = trainer.evaluate()
        print("Test Results (CPU):", test_results)

# Save the model and tokenizer
print("Saving model and tokenizer...")
model_save_path = './saved_model'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model and tokenizer saved to {model_save_path}")

# Optional: Save the training args
training_args.save_pretrained(model_save_path)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/87095 [00:00<?, ? examples/s]

Map:   0%|          | 0/28989 [00:00<?, ? examples/s]

C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\transformers\training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Label validation passed!
Starting training and evaluation...
Number of labels in model: 35
Number of training examples: 87095
Number of test examples: 28989
Training the model...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [5]:
import torch
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, BertTokenizer
from datasets import Dataset
import os
import joblib
import numpy as np

# Debug settings
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load the saved label encoder and data
label_encoder = joblib.load('data/label_encoder.pkl')
num_labels = len(label_encoder.classes_)

# Load test data
X_test = joblib.load('data/X_test.pkl')
y_test_encoded = joblib.load('data/y_test_encoded.pkl')

# Load the pre-trained model and tokenizer
model_path = './saved_model'  # Adjust this path to where your model is saved
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

print(f"Loaded model with {model.config.num_labels} labels")

# Prepare data function
def prepare_dataset(texts, labels, tokenizer, max_length=512):
    return Dataset.from_dict({
        'text': texts['processed_text'].tolist(),
        'labels': labels.tolist()
    }).map(
        lambda x: tokenizer(
            x['text'],
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ),
        batched=True,
        remove_columns=['text']
    )

# Prepare test dataset
test_dataset = prepare_dataset(X_test, y_test_encoded, tokenizer)

# Evaluation arguments
eval_args = TrainingArguments(
    output_dir='./eval_results',
    per_device_eval_batch_size=8,
    no_cuda=False,
    report_to="none"  # Disable wandb logging if you're not using it
)

# Validation function
def validate_data_labels(labels, num_labels):
    unique_labels = np.unique(labels)
    if max(unique_labels) >= num_labels:
        raise ValueError(f"Found label {max(unique_labels)} which is >= num_labels ({num_labels})")
    if min(unique_labels) < 0:
        raise ValueError(f"Found negative label {min(unique_labels)}")
    print("Label validation passed!")

# Validate labels
validate_data_labels(y_test_encoded, num_labels)

# Evaluation with error handling
try:
    print("Starting evaluation...")
    print(f"Number of labels in model: {model.config.num_labels}")
    print(f"Number of test examples: {len(test_dataset)}")
    
    trainer = Trainer(
        model=model,
        args=eval_args,
        eval_dataset=test_dataset,
    )
    
    # Evaluate the model
    print("Evaluating the model...")
    test_results = trainer.evaluate()
    print("Test Results:", test_results)

except Exception as e:
    print(f"Error during evaluation: {str(e)}")
    if "CUDA" in str(e):
        print("CUDA error detected, trying CPU...")
        eval_args.no_cuda = True
        trainer = Trainer(
            model=model,
            args=eval_args,
            eval_dataset=test_dataset,
        )
        test_results = trainer.evaluate()
        print("Test Results (CPU):", test_results)

print("Evaluation complete.")
#abondened due to low compilation speed leaving here so we can try this if needed in the future

Loaded model with 35 labels


Map:   0%|          | 0/28989 [00:00<?, ? examples/s]

Label validation passed!
Starting evaluation...
Number of labels in model: 35
Number of test examples: 28989
Evaluating the model...


KeyboardInterrupt: 

In [1]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import DataLoader, TensorDataset
import os
import joblib
import numpy as np
from tqdm import tqdm

# Enable CUDA optimizations
torch.backends.cudnn.benchmark = True

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load test data
X_test = joblib.load('data/X_test.pkl')
y_test_encoded = joblib.load('data/y_test_encoded.pkl')

# Load the pre-trained model and tokenizer
model_path = './saved_model'
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()

# Modified batch processing function
def process_data_in_batches(texts, labels, tokenizer, max_length=128, batch_size=32):
    all_input_ids = []
    all_attention_masks = []
    
    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
        batch_texts = texts['processed_text'][i:i + batch_size].tolist()
        
        # Ensure consistent padding for all batches
        encodings = tokenizer(
            batch_texts,
            padding='max_length',  # Changed to max_length
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        all_input_ids.append(encodings['input_ids'])
        all_attention_masks.append(encodings['attention_mask'])
    
    # Concatenate all batches
    input_ids = torch.cat(all_input_ids, dim=0)
    attention_masks = torch.cat(all_attention_masks, dim=0)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return input_ids, attention_masks, labels

# Process data with fixed maximum length
print("Processing test data...")
try:
    input_ids, attention_masks, labels = process_data_in_batches(
        X_test, 
        y_test_encoded, 
        tokenizer,
        max_length=128,  # Fixed maximum length
        batch_size=32
    )
    
    # Create DataLoader
    test_dataset = TensorDataset(input_ids, attention_masks, labels)
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False
    )
    
    # Evaluation function
    def evaluate_model(model, dataloader, device):
        model.eval()
        total_eval_loss = 0
        total_correct = 0
        total_samples = 0
        predictions = []
        true_labels = []
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating"):
                batch_input_ids, batch_attention_masks, batch_labels = [b.to(device) for b in batch]
                
                outputs = model(
                    input_ids=batch_input_ids,
                    attention_mask=batch_attention_masks,
                    labels=batch_labels
                )
                
                loss = outputs.loss
                logits = outputs.logits
                
                total_eval_loss += loss.item()
                
                preds = torch.argmax(logits, dim=1)
                total_correct += (preds == batch_labels).sum().item()
                total_samples += batch_labels.size(0)
                
                predictions.extend(preds.cpu().numpy())
                true_labels.extend(batch_labels.cpu().numpy())
        
        avg_loss = total_eval_loss / len(dataloader)
        accuracy = total_correct / total_samples
        
        return {
            'eval_loss': avg_loss,
            'eval_accuracy': accuracy,
            'predictions': predictions,
            'true_labels': true_labels
        }
    
    # Run evaluation
    print("\nStarting evaluation...")
    with torch.cuda.amp.autocast():
        results = evaluate_model(model, test_dataloader, device)
    
    print("\nEvaluation Results:")
    print(f"Loss: {results['eval_loss']:.4f}")
    print(f"Accuracy: {results['eval_accuracy']:.4f}")
    
    # Calculate additional metrics
    from sklearn.metrics import classification_report
    print("\nDetailed Classification Report:")
    print(classification_report(results['true_labels'], results['predictions']))

except Exception as e:
    print(f"Error during evaluation: {str(e)}")
    import traceback
    traceback.print_exc()

print("\nEvaluation complete.")

Using device: cuda
Processing test data...


Tokenizing: 100%|████████████████████████████████████████████████████████████████████| 906/906 [00:21<00:00, 42.56it/s]
C:\Users\Lokesh kumar\AppData\Local\Temp\ipykernel_30040\2933256942.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Starting evaluation...


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 453/453 [01:47<00:00,  4.19it/s]



Evaluation Results:
Loss: 3.6923
Accuracy: 0.0018

Detailed Classification Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        90
           2       0.00      0.00      0.00       719
           3       0.00      0.00      0.00       166
           4       0.50      0.00      0.00      1366
           5       0.00      0.00      0.00        52
           6       0.00      0.97      0.00        39
           7       0.00      0.00      0.00       171
           8       0.00      0.00      0.00      3556
           9       0.00      0.00      0.00       222
          10       0.00      0.00      0.00       187
          11       0.00      0.00      0.00        54
          12       0.00      0.00      0.00      1338
          13       0.00      0.00      0.00       130
          14       0.00      0.00      0.00       763
          15       0.00      0.00      0.00      1827
          16       0.00      0.00      0.00       2

C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.

In [3]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import joblib
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def setup_and_validate():
    """Setup environment and validate data/model"""
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load model and tokenizer
    model_path = './saved_model'
    try:
        tokenizer = BertTokenizer.from_pretrained(model_path)
        model = BertForSequenceClassification.from_pretrained(model_path)
        print(f"Model config: {model.config}")
        print(f"Number of labels in model: {model.num_labels}")
        model.to(device)
        model.eval()
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        raise
    
    # Load test data
    try:
        X_test = joblib.load('data/X_test.pkl')
        y_test_encoded = joblib.load('data/y_test_encoded.pkl')
        print("\nData Loading Success:")
        print(f"Test data shape: {X_test.shape}")
        print(f"Number of test samples: {len(y_test_encoded)}")
        print(f"Unique labels in test set: {np.unique(y_test_encoded)}")
        print("\nLabel distribution:")
        for label, count in enumerate(np.bincount(y_test_encoded)):
            print(f"Label {label}: {count} samples")
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        raise
    
    return device, model, tokenizer, X_test, y_test_encoded

def process_data(texts, labels, tokenizer, max_length=128, batch_size=32):
    """Process and tokenize data"""
    all_input_ids = []
    all_attention_masks = []
    
    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
        batch_texts = texts['processed_text'][i:i + batch_size].tolist()
        
        # Tokenize with padding and truncation
        encodings = tokenizer(
            batch_texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        all_input_ids.append(encodings['input_ids'])
        all_attention_masks.append(encodings['attention_mask'])
    
    # Concatenate all batches
    input_ids = torch.cat(all_input_ids, dim=0)
    attention_masks = torch.cat(all_attention_masks, dim=0)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return input_ids, attention_masks, labels

def evaluate_model(model, dataloader, device, num_labels):
    """Evaluate the model"""
    model.eval()
    total_eval_loss = 0
    predictions = []
    true_labels = []
    all_logits = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch_input_ids, batch_attention_masks, batch_labels = [b.to(device) for b in batch]
            
            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_masks,
                labels=batch_labels
            )
            
            loss = outputs.loss
            logits = outputs.logits
            
            total_eval_loss += loss.item()
            
            # Store predictions and true labels
            batch_preds = torch.argmax(logits, dim=1)
            predictions.extend(batch_preds.cpu().numpy())
            true_labels.extend(batch_labels.cpu().numpy())
            all_logits.extend(logits.cpu().numpy())
            
            # Print sample predictions for the first batch
            if len(predictions) <= batch_input_ids.size(0):
                print("\nSample predictions from first batch:")
                for i in range(min(5, len(batch_preds))):
                    print(f"True: {batch_labels[i].item()}, Predicted: {batch_preds[i].item()}, "
                          f"Confidence: {torch.softmax(logits[i], dim=0).max().item():.4f}")
    
    avg_loss = total_eval_loss / len(dataloader)
    return {
        'eval_loss': avg_loss,
        'predictions': predictions,
        'true_labels': true_labels,
        'logits': all_logits
    }

def plot_confusion_matrix(true_labels, predictions, output_path='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(true_labels, predictions)
    plt.figure(figsize=(15, 15))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(output_path)
    plt.close()

def main():
    try:
        # Setup and validation
        device, model, tokenizer, X_test, y_test_encoded = setup_and_validate()
        
        # Process data
        print("\nProcessing test data...")
        input_ids, attention_masks, labels = process_data(X_test, y_test_encoded, tokenizer)
        
        # Create DataLoader
        test_dataset = TensorDataset(input_ids, attention_masks, labels)
        test_dataloader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=False
        )
        
        # Evaluate
        print("\nStarting evaluation...")
        results = evaluate_model(model, test_dataloader, device, model.num_labels)
        
        # Print results
        print("\nEvaluation Results:")
        print(f"Loss: {results['eval_loss']:.4f}")
        
        # Calculate and print classification report
        print("\nClassification Report:")
        print(classification_report(results['true_labels'], results['predictions']))
        
        # Plot confusion matrix
        plot_confusion_matrix(results['true_labels'], results['predictions'])
        print("\nConfusion matrix has been saved as 'confusion_matrix.png'")
        
        # Calculate and print additional metrics
        accuracy = np.mean(np.array(results['predictions']) == np.array(results['true_labels']))
        print(f"\nOverall Accuracy: {accuracy:.4f}")
        
        # Save predictions
        predictions_df = pd.DataFrame({
            'true_label': results['true_labels'],
            'predicted_label': results['predictions']
        })
        predictions_df.to_csv('predictions.csv', index=False)
        print("\nPredictions saved to 'predictions.csv'")
        
    except Exception as e:
        print(f"\nError during evaluation: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Using device: cuda
Model config: BertConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "./saved_model",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17",
    "18": "LABEL_18",
    "19": "LABEL_19",
    "20": "LABEL_20",
    "21": "LABEL_21",
    "22": "LABEL_22",
    "23": "LABEL_23",
    "24": "LABEL_24",
    "25": "LABEL_25",
    "26": "LABEL_26",
    "27": "LABEL_27",
    "28": "LABEL_28",
  

Tokenizing: 100%|████████████████████████████████████████████████████████████████████| 906/906 [00:18<00:00, 49.59it/s]



Starting evaluation...


Evaluating:   0%|                                                                      | 1/906 [00:00<09:40,  1.56it/s]


Sample predictions from first batch:
True: 8, Predicted: 6, Confidence: 0.0769
True: 30, Predicted: 6, Confidence: 0.0529
True: 15, Predicted: 6, Confidence: 0.0770
True: 25, Predicted: 6, Confidence: 0.0725
True: 18, Predicted: 6, Confidence: 0.0587


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 906/906 [07:36<00:00,  1.99it/s]
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with


Evaluation Results:
Loss: 3.6923

Classification Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        90
           2       0.00      0.00      0.00       719
           3       0.00      0.00      0.00       166
           4       0.50      0.00      0.00      1366
           5       0.00      0.00      0.00        52
           6       0.00      0.97      0.00        39
           7       0.00      0.00      0.00       171
           8       0.00      0.00      0.00      3556
           9       0.00      0.00      0.00       222
          10       0.00      0.00      0.00       187
          11       0.00      0.00      0.00        54
          12       0.00      0.00      0.00      1338
          13       0.00      0.00      0.00       130
          14       0.00      0.00      0.00       763
          15       0.00      0.00      0.00      1827
          16       0.00      0.00      0.00       200
          17       0.00

In [6]:
import torch
from transformers import (
    BertForSequenceClassification, 
    BertTokenizer, 
    Trainer, 
    TrainingArguments
)
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score
)
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import Dataset

def setup_and_validate():
    """Setup environment and validate data/model"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model_path = './saved_model'
    try:
        tokenizer = BertTokenizer.from_pretrained(model_path)
        model = BertForSequenceClassification.from_pretrained(model_path)
        print(f"Model config: {model.config}")
        print(f"Number of labels in model: {model.num_labels}")
        model.to(device)
        model.eval()
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        raise
    
    try:
        X_test = joblib.load('data/X_test.pkl')
        y_test_encoded = joblib.load('data/y_test_encoded.pkl')
        
        # Adjust labels if necessary (assuming test labels start from 1)
        y_test_encoded = y_test_encoded - 1
        
        print("\nData Loading Success:")
        print(f"Test data shape: {X_test.shape}")
        print(f"Number of test samples: {len(y_test_encoded)}")
        print(f"Unique labels in test set: {np.unique(y_test_encoded)}")
        
        print("\nLabel distribution in test set:")
        for label, count in enumerate(np.bincount(y_test_encoded)):
            print(f"Label {label}: {count} samples")
        
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        raise
    
    return device, model, tokenizer, X_test, y_test_encoded

def process_data(texts, labels, tokenizer, max_length=128):
    """Process and tokenize data"""
    dataset = Dataset.from_dict({"text": texts['processed_text'], "labels": labels})
    
    def tokenize_function(examples):
        return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=max_length)
    
    print("\nProcessing test data...")
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    return tokenized_dataset

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def plot_confusion_matrix(true_labels, predictions, output_path='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(true_labels, predictions)
    plt.figure(figsize=(20, 20))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(output_path)
    plt.close()

def analyze_predictions(true_labels, predictions, test_texts, n_samples=5):
    """Analyze some predictions to understand model behavior"""
    print("\nSample Predictions Analysis:")
    indices = np.random.randint(0, len(true_labels), n_samples)
    
    for idx in indices:
        print(f"\nText: {test_texts.iloc[idx]['processed_text'][:100]}...")
        print(f"True Label: {true_labels[idx]}")
        print(f"Predicted Label: {predictions[idx]}")

def main():
    try:
        device, model, tokenizer, X_test, y_test_encoded = setup_and_validate()
        
        test_dataset = process_data(X_test, y_test_encoded, tokenizer)
        
        print("\nSetting up Trainer for evaluation...")
        training_args = TrainingArguments(
            output_dir='./results',
            per_device_eval_batch_size=64,
            logging_dir='./logs',
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            compute_metrics=compute_metrics,
        )
        
        print("\nStarting evaluation...")
        eval_results = trainer.evaluate(test_dataset)
        
        print("\nEvaluation Results:")
        print(eval_results)
        
        print("\nGenerating predictions...")
        predictions = trainer.predict(test_dataset)
        preds = predictions.predictions.argmax(-1)
        
        print("\nClassification Report:")
        print(classification_report(y_test_encoded, preds))
        
        plot_confusion_matrix(y_test_encoded, preds)
        print("\nConfusion matrix has been saved as 'confusion_matrix.png'")
        
        # Analyze some predictions
        analyze_predictions(y_test_encoded, preds, X_test)
        
        accuracy = np.mean(preds == y_test_encoded)
        print(f"\nOverall Accuracy: {accuracy:.4f}")
        
        # Save predictions
        predictions_df = pd.DataFrame({
            'text': X_test['processed_text'],
            'true_label': y_test_encoded,
            'predicted_label': preds
        })
        predictions_df.to_csv('predictions.csv', index=False)
        print("\nPredictions saved to 'predictions.csv'")
        
    except Exception as e:
        print(f"\nError during evaluation: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Using device: cuda
Model config: BertConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "./saved_model",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17",
    "18": "LABEL_18",
    "19": "LABEL_19",
    "20": "LABEL_20",
    "21": "LABEL_21",
    "22": "LABEL_22",
    "23": "LABEL_23",
    "24": "LABEL_24",
    "25": "LABEL_25",
    "26": "LABEL_26",
    "27": "LABEL_27",
    "28": "LABEL_28",
  

Map:   0%|          | 0/28989 [00:00<?, ? examples/s]


Setting up Trainer for evaluation...

Starting evaluation...


C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Evaluation Results:
{'eval_loss': 3.6426620483398438, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.0037945427575977095, 'eval_f1': 6.269578823676954e-05, 'eval_precision': 3.277486496417784e-05, 'eval_recall': 0.0037945427575977095, 'eval_runtime': 2180.5211, 'eval_samples_per_second': 13.295, 'eval_steps_per_second': 0.208}

Generating predictions...


C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Lokesh kumar\Cyberfraud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.


Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        90
           1       0.00      0.00      0.00       719
           2       0.00      0.00      0.00       166
           3       0.00      0.00      0.00      1366
           4       0.00      0.00      0.00        52
           5       0.00      0.00      0.00        39
           6       0.00      0.63      0.01       171
           7       0.00      0.00      0.00      3556
           8       0.00      0.00      0.00       222
           9       0.00      0.00      0.00       187
          10       0.00      0.00      0.00        54
          11       0.00      0.00      0.00      1338
          12       0.00      0.00      0.00       130
          13       0.00      0.00      0.00       763
          14       0.00      0.00      0.00      1827
          15       0.00      0.00      0.00       200
          16       0.00      0.00      0.00        13
   